# AMEX Enterprise Credit Risk Platform
## Notebook 09 — MLOps: Model Registry, CI/CD, Latency Benchmark & Model Card
### Phase 1 · Problem Statement 1: Credit Scoring / PD Prediction

CRISP-DM stage: **Deployment**. Notebook 9 of 18. Depends on Notebooks 01 and 05; Notebook 07's model-risk findings are used opportunistically if present, same convention as every notebook since 07.

**What is genuinely computed here.** Everything in this notebook is either a real, live measurement or a real, valid generated artifact — there is no financial or business-scenario content here, so the MEASURED/ASSUMPTION distinction used in Notebooks 08 and 14 mostly doesn't apply:

- **Inference latency** (model load time, p50/p95/p99 per-row scoring latency, throughput) is benchmarked live, this run, on the real champion model and real holdout data — not estimated.
- **The model's file hash** (SHA-256) is computed live from the real saved `.joblib` file, giving the registry entry genuine integrity verification.
- **The CI/CD pipeline definition** is a real, valid GitHub Actions YAML workflow (lint → validate model floor → package → deploy-stub) — a template you can drop into `.github/workflows/` as-is, not a mockup.
- **The environment snapshot** (`requirements.txt`) captures the real installed versions of every library this platform actually imports, via `importlib.metadata`, not a hand-typed list.

**Deliverables:** `model_registry.json` (versioned, appended to on every run), `requirements.txt`, `ci_cd_pipeline.yml`, `retraining_trigger_policy.json`, `deployment_readiness_checklist.csv`, 2 charts, and `MLOps_Model_Card_And_Deployment_Report.docx` (a standard ML model card).

**Run the single code cell below, once.** Idempotent for every file except `model_registry.json`, which is deliberately append-only (a real registry's whole point is to keep every past version, not overwrite its own history) — every other output file is overwritten in place on re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 01, 05 (07 OPTIONAL)
# =============================================================================
import os
import sys
import csv
import json
import time
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01, 05 (07 Optional)")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB04_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_04_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB07_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_07_summary.json"  # optional

for _p, _fix in [(CONFIG_PATH, "run 01_business_understanding.ipynb first"),
                  (NB04_SUMMARY_PATH, "run 04_feature_engineering.ipynb first")]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB04_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB04_SUMMARY = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (_resource_limits.get("warp_thread_count") or PROJECT_CONFIG.get("warp_thread_count")
                      or PROJECT_CONFIG["hardware"].get("logical_cores_detected"))

MODEL_DEV_DIR = PILLAR_DIRS["model_development"]
MODELS_SUBDIR = MODEL_DEV_DIR / "models"
MLOPS_DIR = PILLAR_DIRS["mlops"]
MLOPS_DIR.mkdir(parents=True, exist_ok=True)

TEST_SPLIT_ENG_PATH = Path(NB04_SUMMARY["output_files"]["test_split_engineered.csv"])
MODEL_COMPARISON_PATH = MODEL_DEV_DIR / "model_comparison.csv"
PREPROCESSING_PATH = MODELS_SUBDIR / "preprocessing_artifacts.joblib"

NB05_SUMMARY = None
if NB05_SUMMARY_PATH.exists():
    with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
        NB05_SUMMARY = json.load(f)
    CHAMPION_NAME = NB05_SUMMARY["champion_model"]
    CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
    _champion_source = NB05_SUMMARY_PATH.name
elif MODEL_COMPARISON_PATH.exists():
    with open(MODEL_COMPARISON_PATH, "r", encoding="utf-8", newline="") as _f:
        _cmp_rows = list(csv.DictReader(_f))
    if not _cmp_rows or "model" not in _cmp_rows[0] or "holdout_amex_metric" not in _cmp_rows[0]:
        raise RuntimeError(f"{MODEL_COMPARISON_PATH} is missing expected columns. Fix: re-run Notebook 05.")
    _champion_row = max(_cmp_rows, key=lambda r: float(r["holdout_amex_metric"]))
    CHAMPION_NAME = _champion_row["model"]
    CHAMPION_METRICS = {k: (float(v) if k != "model" else v) for k, v in _champion_row.items()}
    _champion_source = f"{MODEL_COMPARISON_PATH.name} (fallback)"
else:
    raise FileNotFoundError(f"Neither {NB05_SUMMARY_PATH} nor {MODEL_COMPARISON_PATH} found.\n"
                             f"Fix: run 05_model_development.ipynb first.")

NB07_SUMMARY = None
if NB07_SUMMARY_PATH.exists():
    with open(NB07_SUMMARY_PATH, "r", encoding="utf-8") as f:
        NB07_SUMMARY = json.load(f)

CHAMPION_MODEL_PATH = MODELS_SUBDIR / f"{CHAMPION_NAME}.joblib"
for _p in (TEST_SPLIT_ENG_PATH, CHAMPION_MODEL_PATH, PREPROCESSING_PATH):
    if not _p.exists():
        raise FileNotFoundError(f"Required file not found: {_p}\nFix: re-run 05_model_development.ipynb.")

print(f"Champion model          : {CHAMPION_NAME}  (identified from: {_champion_source})")
print(f"Notebook 07 MRM output   : {'found -- will inform retraining trigger policy' if NB07_SUMMARY else 'not found -- using default stated thresholds'}")
print(f"MLOps outputs will be written under: {MLOPS_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION, LIBRARY IMPORTS & ADAPTIVE RAM CEILING
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration, Library Imports & Adaptive RAM Ceiling")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from docx import Document
    from docx.shared import Inches
except ImportError:
    missing.append("python-docx")
try:
    from importlib import metadata as importlib_metadata
except ImportError:
    missing.append("importlib_metadata")

if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) + "\n"
                       f"Fix: pip install {' '.join(missing)}")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()

_live_vm = psutil.virtual_memory()
LIVE_AVAILABLE_RAM_BYTES = _live_vm.available
ADAPTIVE_RAM_FRACTION = _resource_limits.get("ram_fraction_cap", 0.90)
MAX_RAM_BYTES = int(LIVE_AVAILABLE_RAM_BYTES * ADAPTIVE_RAM_FRACTION)

print(f"WARP_THREAD_COUNT                : {WARP_THREAD_COUNT}")
print(f"Live available RAM right now      : {LIVE_AVAILABLE_RAM_BYTES / 1e9:.1f} GB")
print(f"Adaptive RAM ceiling (this run)   : {MAX_RAM_BYTES / 1e9:.2f} GB")
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: MODEL FILE INTEGRITY (SHA-256) & LIVE INFERENCE LATENCY BENCHMARK
# =============================================================================
_section("SECTION 3: Model File Integrity & Live Inference Latency Benchmark")

# --- Every number in this section is a REAL, live measurement -- the model's
#     file hash is computed by reading its actual bytes, and every latency
#     figure comes from actually timing predict_proba calls on the real
#     champion model against the real holdout data, not an estimate. ---


def _sha256_of_file(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


_t0 = time.time()
CHAMPION_MODEL_SHA256 = _sha256_of_file(CHAMPION_MODEL_PATH)
_hash_seconds = time.time() - _t0
CHAMPION_MODEL_FILE_SIZE_MB = CHAMPION_MODEL_PATH.stat().st_size / 1e6

_t0 = time.time()
champion_model = joblib.load(CHAMPION_MODEL_PATH)
preprocessing_artifacts = joblib.load(PREPROCESSING_PATH)
MODEL_LOAD_SECONDS = time.time() - _t0

label_encoders = preprocessing_artifacts["label_encoders"]
feature_medians = preprocessing_artifacts["feature_medians"]
scaler = preprocessing_artifacts["scaler"]
all_feature_cols = preprocessing_artifacts["all_feature_cols"]
categorical_encode_cols = preprocessing_artifacts["categorical_encode_cols"]
numeric_feature_cols = preprocessing_artifacts["numeric_feature_cols"]
champion_uses_scaled = CHAMPION_NAME == "logistic_regression"

print(f"Model file          : {CHAMPION_MODEL_PATH.name}  ({CHAMPION_MODEL_FILE_SIZE_MB:.2f} MB)")
print(f"SHA-256 (computed, live): {CHAMPION_MODEL_SHA256}  (hashed in {_hash_seconds:.2f}s)")
print(f"Model load time (measured): {MODEL_LOAD_SECONDS:.3f}s")

SPLIT_CSV_SCHEMA = {"customer_ID": pl.Utf8, "target": pl.Int8}
for _c in categorical_encode_cols:
    SPLIT_CSV_SCHEMA[_c] = pl.Utf8
for _c in numeric_feature_cols:
    SPLIT_CSV_SCHEMA[_c] = pl.Float32

holdout_pl = pl.read_csv(str(TEST_SPLIT_ENG_PATH), schema_overrides=SPLIT_CSV_SCHEMA)
_inf_clean_exprs = [
    pl.when(pl.col(c).is_infinite() | pl.col(c).is_nan()).then(None).otherwise(pl.col(c)).cast(pl.Float32).alias(c)
    for c in numeric_feature_cols
]
holdout_pl = holdout_pl.with_columns(_inf_clean_exprs)
for c in categorical_encode_cols:
    holdout_pl = holdout_pl.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))
    _mapping = {cat: i for i, cat in enumerate(label_encoders[c]["classes"])}
    holdout_pl = holdout_pl.with_columns(pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))
_impute_exprs = [pl.col(c).fill_null(feature_medians[c]) for c in numeric_feature_cols]
holdout_pl = holdout_pl.with_columns(_impute_exprs)

X_holdout = holdout_pl.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
_train_mean, _train_std = scaler["mean"], scaler["std"]
X_holdout_scaled = (X_holdout - _train_mean) / _train_std
Xc_holdout = X_holdout_scaled if champion_uses_scaled else X_holdout

# --- Single-row latency benchmark: score one customer at a time, repeated,
#     to measure realistic real-time-scoring-API latency (p50/p95/p99) --
#     distinct from the batch-throughput benchmark below. ---
N_LATENCY_SAMPLES = min(200, Xc_holdout.shape[0])
_rng = np.random.RandomState(RANDOM_SEED)
_sample_idx = _rng.choice(Xc_holdout.shape[0], size=N_LATENCY_SAMPLES, replace=False)
_single_row_latencies_ms = []
for _i in _sample_idx:
    _row = Xc_holdout[_i:_i + 1]
    _t0 = time.perf_counter()
    _ = champion_model.predict_proba(_row)
    _single_row_latencies_ms.append((time.perf_counter() - _t0) * 1000.0)
_single_row_latencies_ms = np.array(_single_row_latencies_ms)

# --- Batch throughput benchmark: score the whole holdout split at once. ---
_t0 = time.time()
_ = champion_model.predict_proba(Xc_holdout)
_batch_seconds = time.time() - _t0
BATCH_THROUGHPUT_ROWS_PER_SEC = Xc_holdout.shape[0] / _batch_seconds if _batch_seconds > 0 else float("inf")

latency_summary = {
    "n_latency_samples": N_LATENCY_SAMPLES,
    "single_row_p50_ms": round(float(np.percentile(_single_row_latencies_ms, 50)), 4),
    "single_row_p95_ms": round(float(np.percentile(_single_row_latencies_ms, 95)), 4),
    "single_row_p99_ms": round(float(np.percentile(_single_row_latencies_ms, 99)), 4),
    "single_row_max_ms": round(float(_single_row_latencies_ms.max()), 4),
    "batch_rows_scored": int(Xc_holdout.shape[0]),
    "batch_seconds": round(_batch_seconds, 4),
    "batch_throughput_rows_per_sec": round(BATCH_THROUGHPUT_ROWS_PER_SEC, 1),
    "model_load_seconds": round(MODEL_LOAD_SECONDS, 4),
}

print(f"\nSingle-row scoring latency (measured, {N_LATENCY_SAMPLES} samples):")
print(f"  p50: {latency_summary['single_row_p50_ms']:.3f} ms   p95: {latency_summary['single_row_p95_ms']:.3f} ms   "
      f"p99: {latency_summary['single_row_p99_ms']:.3f} ms   max: {latency_summary['single_row_max_ms']:.3f} ms")
print(f"Batch throughput (measured, {latency_summary['batch_rows_scored']:,} rows): "
      f"{latency_summary['batch_throughput_rows_per_sec']:,.0f} rows/sec")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: MODEL REGISTRY (VERSIONED, APPEND-ONLY -- REAL PAST VERSIONS KEPT)
# =============================================================================
_section("SECTION 4: Model Registry (Versioned, Append-Only)")

# --- A real registry's whole point is to keep every past version, not
#     overwrite its own history -- this is the one file in this notebook that
#     is deliberately APPEND-only rather than idempotent-overwrite. Each
#     re-run appends a new version entry only if the model's file hash has
#     actually changed since the last recorded entry (re-running against an
#     unchanged model does not create a duplicate version). ---
registry_path = MLOPS_DIR / "model_registry.json"
if registry_path.exists():
    with open(registry_path, "r", encoding="utf-8") as f:
        registry = json.load(f)
else:
    registry = {"entries": []}

_existing_hashes = {e["sha256"] for e in registry["entries"] if e.get("model_name") == CHAMPION_NAME}
if CHAMPION_MODEL_SHA256 in _existing_hashes:
    print(f"Model hash {CHAMPION_MODEL_SHA256[:12]}... already registered for '{CHAMPION_NAME}' -- "
          f"no new version appended (the saved model file has not changed since the last registry entry).")
    _new_version = next(e["version"] for e in registry["entries"] if e["sha256"] == CHAMPION_MODEL_SHA256)
else:
    _prior_versions = [e["version"] for e in registry["entries"] if e.get("model_name") == CHAMPION_NAME]
    _new_version = (max(_prior_versions) + 1) if _prior_versions else 1
    registry["entries"].append({
        "model_name": CHAMPION_NAME, "version": _new_version,
        "registered_at_utc": datetime.now(timezone.utc).isoformat(),
        "sha256": CHAMPION_MODEL_SHA256, "file_size_mb": round(CHAMPION_MODEL_FILE_SIZE_MB, 3),
        "holdout_auc": CHAMPION_METRICS.get("holdout_auc"), "holdout_amex_metric": CHAMPION_METRICS.get("holdout_amex_metric"),
        "holdout_top4pct_capture": CHAMPION_METRICS.get("holdout_top4pct_capture"),
        "risk_tier": NB07_SUMMARY.get("risk_tier") if NB07_SUMMARY else None,
        "latency_p50_ms": latency_summary["single_row_p50_ms"], "latency_p99_ms": latency_summary["single_row_p99_ms"],
        "batch_throughput_rows_per_sec": latency_summary["batch_throughput_rows_per_sec"],
    })
    with open(registry_path, "w", encoding="utf-8") as f:
        json.dump(registry, f, indent=2)
    print(f"Appended NEW registry entry: '{CHAMPION_NAME}' version {_new_version}")

print(f"Registry now contains {len(registry['entries'])} total version(s) across all models ever registered.")
print(f"\u2705 Registry at -> {registry_path}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: ENVIRONMENT SNAPSHOT (REAL INSTALLED PACKAGE VERSIONS)
# =============================================================================
_section("SECTION 5: Environment Snapshot (requirements.txt)")

# --- Captures the REAL installed version of every library this platform's
#     notebooks actually import, via importlib.metadata -- not a hand-typed
#     guess. A package that isn't installed in THIS environment is reported
#     honestly as "not installed here" rather than silently omitted. ---
_tracked_packages = ["polars", "numpy", "pandas", "scikit-learn", "xgboost", "lightgbm", "catboost",
                      "shap", "lime", "matplotlib", "psutil", "joblib", "python-docx", "scipy", "openpyxl"]
_pkg_versions = {}
for _pkg in _tracked_packages:
    try:
        _pkg_versions[_pkg] = importlib_metadata.version(_pkg)
    except importlib_metadata.PackageNotFoundError:
        _pkg_versions[_pkg] = None

requirements_path = MLOPS_DIR / "requirements.txt"
with open(requirements_path, "w", encoding="utf-8") as f:
    f.write(f"# Auto-generated by 09_mlops.ipynb -- real installed versions on this machine, {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
    f.write(f"# Python {sys.version.split()[0]}\n\n")
    for _pkg, _ver in _pkg_versions.items():
        if _ver:
            f.write(f"{_pkg}=={_ver}\n")
        else:
            f.write(f"# {_pkg}  -- not installed in this environment; install if your deployment target needs it\n")

for _pkg, _ver in _pkg_versions.items():
    print(f"  {_pkg:<16}: {_ver if _ver else 'not installed here'}")
print(f"\u2705 Saved -> {requirements_path}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: CI/CD PIPELINE DEFINITION (REAL, VALID GITHUB ACTIONS YAML)
# =============================================================================
_section("SECTION 6: CI/CD Pipeline Definition")

# --- A real, valid GitHub Actions workflow -- drop this into
#     .github/workflows/ as-is. It is a TEMPLATE (no actual cloud deployment
#     target is configured, since this platform has none), but every step is
#     a real, runnable command against this repository's actual structure.
#     The model-floor check references Notebook 01's stated deployment floor
#     (AMEX metric >= 0.75) so a regression is caught in CI, not after deploy. ---
_deployment_floor = 0.75  # matches Notebook 01's stated Risk Appetite deployment floor
CI_CD_YAML = f"""name: AMEX Credit Risk Platform CI/CD

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  lint-and-validate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install dependencies
        run: pip install -r MLOps/requirements.txt
      - name: Lint notebooks (nbqa/flake8 or equivalent)
        run: echo "Add your notebook-lint command here (e.g. nbqa flake8 notebooks/)"

  validate-model-floor:
    needs: lint-and-validate
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Check champion model against deployment floor
        run: |
          python -c "
          import json
          with open('artifacts/notebook_05_summary.json') as f:
              summary = json.load(f)
          amex_metric = summary['champion_metrics']['holdout_amex_metric']
          floor = {_deployment_floor}
          assert amex_metric >= floor, f'Champion AMEX metric {{amex_metric:.4f}} is below the deployment floor {{floor}}'
          print(f'PASS: champion AMEX metric {{amex_metric:.4f}} >= floor {{floor}}')
          "

  package-and-deploy-stub:
    needs: validate-model-floor
    runs-on: ubuntu-latest
    if: github.ref == 'refs/heads/main'
    steps:
      - uses: actions/checkout@v4
      - name: Package model artifacts
        run: echo "Add real packaging step here (e.g. docker build -t amex-pd-api:$GITHUB_SHA .)"
      - name: Deploy (stub -- configure your real deployment target)
        run: echo "Add real deployment step here (e.g. push to your container registry / call your platform's deploy API)"
"""

ci_cd_path = MLOPS_DIR / "ci_cd_pipeline.yml"
with open(ci_cd_path, "w", encoding="utf-8") as f:
    f.write(CI_CD_YAML)

# Sanity-check the generated YAML actually parses (catches template bugs before delivery)
try:
    import yaml as _yaml
    _yaml.safe_load(CI_CD_YAML)
    print("Generated pipeline YAML parses successfully (yaml.safe_load).")
except ImportError:
    print("(pyyaml not installed in this environment -- skipping the parse self-check; the file was still "
          "written and is standard GitHub Actions YAML.)")

print(f"\u2705 Saved -> {ci_cd_path}")
print(f"Deployment floor referenced (from Notebook 01's stated Risk Appetite Statement): AMEX metric >= {_deployment_floor}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: AUTOMATED RETRAINING TRIGGER POLICY
# =============================================================================
_section("SECTION 7: Automated Retraining Trigger Policy")

# --- Stated policy (documented, not "discovered"), same convention as
#     Notebook 07's risk-tiering rubric -- but the trigger STATUS reported
#     below is real: it reads Notebook 07's actual PSI/rank-ordering findings
#     when available, rather than assuming a value. ---
retraining_policy = {
    "trigger_1_population_stability": {
        "rule": "PSI > 0.25 on any top-30 feature (Notebook 07) triggers mandatory review",
        "current_status": (f"{NB07_SUMMARY.get('psi_significant_shift_features', 0)} feature(s) currently over threshold"
                            if NB07_SUMMARY else "Notebook 07 has not been run yet -- status unknown"),
    },
    "trigger_2_rank_ordering": {
        "rule": "Any decile-to-decile rank-ordering inversion (Notebook 07) triggers review",
        "current_status": (f"{NB07_SUMMARY.get('rank_ordering_inversions', 0)} inversion(s) currently found"
                            if NB07_SUMMARY else "Notebook 07 has not been run yet -- status unknown"),
    },
    "trigger_3_performance_floor": {
        "rule": f"Champion holdout AMEX metric falling below {_deployment_floor} (Notebook 01's stated floor) triggers immediate retraining",
        "current_status": f"Current champion AMEX metric: {CHAMPION_METRICS.get('holdout_amex_metric', 'n/a')}",
    },
    "trigger_4_scheduled_cadence": {
        "rule": "Quarterly scheduled retraining regardless of drift signals (stated policy)",
        "current_status": "Scheduling is an operational/CI concern outside this notebook's scope -- see ci_cd_pipeline.yml",
    },
}
retraining_policy_path = MLOPS_DIR / "retraining_trigger_policy.json"
with open(retraining_policy_path, "w", encoding="utf-8") as f:
    json.dump(retraining_policy, f, indent=2)

for _k, _v in retraining_policy.items():
    print(f"\n{_k}:")
    print(f"  Rule   : {_v['rule']}")
    print(f"  Status : {_v['current_status']}")
print(f"\n\u2705 Saved -> {retraining_policy_path}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: DEPLOYMENT READINESS CHECKLIST
# =============================================================================
_section("SECTION 8: Deployment Readiness Checklist")

deployment_checklist = [
    {"dimension": "Model Registered with Integrity Hash", "status": "Pass", "evidence": f"model_registry.json, SHA-256 {CHAMPION_MODEL_SHA256[:16]}..."},
    {"dimension": "Inference Latency Benchmarked", "status": "Pass",
     "evidence": f"p99 {latency_summary['single_row_p99_ms']:.2f}ms single-row, {latency_summary['batch_throughput_rows_per_sec']:,.0f} rows/sec batch"},
    {"dimension": "Environment Snapshot Captured", "status": "Pass", "evidence": "requirements.txt, this run"},
    {"dimension": "CI/CD Pipeline Defined", "status": "Pass", "evidence": "ci_cd_pipeline.yml, this run"},
    {"dimension": "Automated Retraining Triggers Defined", "status": "Pass", "evidence": "retraining_trigger_policy.json, this run"},
    {"dimension": "Model Risk Validation (SR 11-7 / MRM)", "status": "Pass" if NB07_SUMMARY else "Not Yet Completed",
     "evidence": f"Risk tier: {NB07_SUMMARY.get('risk_tier')}" if NB07_SUMMARY else "Run 07_model_risk_management.ipynb"},
    {"dimension": "Meets Stated Deployment Floor", "status": "Pass" if CHAMPION_METRICS.get("holdout_amex_metric", 0) >= _deployment_floor else "Fail",
     "evidence": f"AMEX metric {CHAMPION_METRICS.get('holdout_amex_metric')} vs floor {_deployment_floor}"},
    {"dimension": "Scoring API Implementation (FastAPI)", "status": "Not Yet Completed", "evidence": "Deferred to Notebook 10"},
    {"dimension": "Containerization (Docker)", "status": "Not Yet Completed", "evidence": "Deferred to Notebook 11"},
    {"dimension": "Production Monitoring Configured", "status": "Not Yet Completed", "evidence": "Deferred to Notebook 12"},
]
deployment_df = pd.DataFrame(deployment_checklist)
deployment_checklist_path = MLOPS_DIR / "deployment_readiness_checklist.csv"
deployment_df.to_csv(deployment_checklist_path, index=False)
print(deployment_df.to_string(index=False))
print(f"\u2705 Saved -> {deployment_checklist_path}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: CHARTS -- LATENCY DISTRIBUTION & THROUGHPUT
# =============================================================================
_section("SECTION 9: Charts")

PROBLEM_NAME = "Phase 1 \u00b7 Problem 1 -- Credit Scoring / PD Prediction"
VIZ = {"surface": "#fcfcfb", "text_primary": "#0b0b0b", "text_secondary": "#52514e", "grid": "#e3e2dd",
       "cat_blue": "#2a78d6", "cat_red": "#e34948", "cat_green": "#3a9e5f"}


def _style_axes(ax):
    ax.set_facecolor(VIZ["surface"]); ax.figure.set_facecolor(VIZ["surface"])
    ax.grid(axis="y", color=VIZ["grid"], linewidth=0.8, zorder=0); ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(VIZ["grid"])
    ax.tick_params(colors=VIZ["text_secondary"], labelsize=9)
    ax.title.set_color(VIZ["text_primary"])
    ax.xaxis.label.set_color(VIZ["text_secondary"]); ax.yaxis.label.set_color(VIZ["text_secondary"])


fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)
ax.hist(_single_row_latencies_ms, bins=30, color=VIZ["cat_blue"], zorder=3)
for _pct, _color in [("single_row_p50_ms", VIZ["cat_green"]), ("single_row_p99_ms", VIZ["cat_red"])]:
    ax.axvline(latency_summary[_pct], color=_color, linestyle="--", linewidth=1.4, zorder=4,
               label=f"{_pct.replace('single_row_', '').replace('_ms', '')} = {latency_summary[_pct]:.2f} ms")
_style_axes(ax)
ax.set_xlabel("Single-row scoring latency (ms)")
ax.set_ylabel("Number of samples")
ax.set_title(f"{PROBLEM_NAME}\nInference Latency Distribution, Champion ({CHAMPION_NAME}, measured)", fontsize=11)
ax.legend(frameon=False)
fig.tight_layout()
chart1_path = MLOPS_DIR / "mlops_latency_distribution_chart.png"
fig.savefig(chart1_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart1_path}")

fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=150)
_bars = ax.bar(["Model Load", "Batch Scoring\n(full holdout)"],
                [latency_summary["model_load_seconds"], latency_summary["batch_seconds"]],
                color=[VIZ["cat_blue"], VIZ["cat_green"]], zorder=3)
ax.bar_label(_bars, fmt="%.3fs", padding=3, fontsize=9, color=VIZ["text_primary"])
_style_axes(ax)
ax.set_ylabel("Seconds")
ax.set_title(f"{PROBLEM_NAME}\nModel Load Time vs. Batch Scoring Time (measured)", fontsize=11)
fig.tight_layout()
chart2_path = MLOPS_DIR / "mlops_load_vs_batch_chart.png"
fig.savefig(chart2_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart2_path}")

print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: MODEL CARD & DEPLOYMENT REPORT (WORD DOCUMENT)
# =============================================================================
_section("SECTION 10: Model Card & Deployment Report (Word Document)")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = str(k).replace("_", " ").title()
        row[1].text = str(v)
    return table


report = Document()
report.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
report.add_paragraph("MLOps: Model Card & Deployment Report -- Notebook 09")
report.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(report, "1. Model Details", level=1)
_add_kv_table(report, {
    "model_name": CHAMPION_NAME, "version": _new_version, "sha256": CHAMPION_MODEL_SHA256,
    "file_size_mb": round(CHAMPION_MODEL_FILE_SIZE_MB, 3), "feature_count": len(all_feature_cols),
})

_add_heading(report, "2. Intended Use", level=1)
report.add_paragraph(
    "Predicts probability of default (PD) for American Express card customers, to support credit "
    "underwriting, pricing, and collections-prioritization decisions. NOT intended for use on applicant "
    "populations outside the training distribution (see Notebook 01's reject-inference limitation) without "
    "re-validation."
)

_add_heading(report, "3. Training Data & Evaluation", level=1)
_add_kv_table(report, {
    "holdout_auc": CHAMPION_METRICS.get("holdout_auc"), "holdout_amex_metric": CHAMPION_METRICS.get("holdout_amex_metric"),
    "holdout_top4pct_capture": CHAMPION_METRICS.get("holdout_top4pct_capture"),
    "model_risk_tier": NB07_SUMMARY.get("risk_tier") if NB07_SUMMARY else "Pending Notebook 07",
})

_add_heading(report, "4. Inference Performance (Measured, This Run)", level=1)
report.add_picture(str(chart1_path), width=Inches(6.0))
_add_kv_table(report, {k: v for k, v in latency_summary.items()})
report.add_picture(str(chart2_path), width=Inches(5.5))

_add_heading(report, "5. Ethical Considerations & Caveats", level=1)
for _d in [
    "The AMEX dataset is accepts-only (no rejected-applicant data) -- this model cannot by itself validate "
    "how a never-approved population would perform (Notebook 01).",
    "Fair-lending / disparate-impact testing is tracked in Notebook 07's governance checklist, not repeated here.",
    "This model card documents the CHAMPION model identified by Notebook 05's holdout AMEX metric ranking -- "
    "re-run Notebook 05 and this notebook together if the champion changes.",
]:
    report.add_paragraph(_d, style="List Bullet")

_add_heading(report, "6. Deployment Readiness Checklist", level=1)
_dep_table = report.add_table(rows=1, cols=3)
_dep_table.style = "Light Grid Accent 1"
_hdr = _dep_table.rows[0].cells
_hdr[0].text, _hdr[1].text, _hdr[2].text = "Dimension", "Status", "Evidence"
for r in deployment_checklist:
    c = _dep_table.add_row().cells
    c[0].text, c[1].text, c[2].text = r["dimension"], r["status"], r["evidence"]

_add_heading(report, "7. Retraining Trigger Policy", level=1)
for _k, _v in retraining_policy.items():
    report.add_paragraph(f"{_k}: {_v['rule']}", style="List Bullet")
    report.add_paragraph(f"  Current status: {_v['current_status']}")

report_path = MLOPS_DIR / "MLOps_Model_Card_And_Deployment_Report.docx"
report.save(str(report_path))
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 11: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("SHA-256 hash is 64 hex characters", len(CHAMPION_MODEL_SHA256) == 64)
_check("latency samples are all non-negative", bool((_single_row_latencies_ms >= 0).all()))
_check("p50 <= p95 <= p99 (latency percentiles are ordered)",
       latency_summary["single_row_p50_ms"] <= latency_summary["single_row_p95_ms"] <= latency_summary["single_row_p99_ms"])
_check("batch throughput is positive", latency_summary["batch_throughput_rows_per_sec"] > 0)
_check("registry contains at least 1 entry", len(registry["entries"]) >= 1)
_check("deployment checklist covers 10 dimensions", len(deployment_df) == 10, f"({len(deployment_df)})")

_expected_files = [registry_path, requirements_path, ci_cd_path, retraining_policy_path,
                    deployment_checklist_path, chart1_path, chart2_path, report_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 09 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 09 checks passed.")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 12: Resource / Performance Report")

_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "adaptive_ram_ceiling_gb": round(MAX_RAM_BYTES / 1e9, 2),
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
    "inference_latency_summary": latency_summary,
}
performance_report_path = ARTIFACTS_DIR / "notebook_09_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)
print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)")
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: WRITE NOTEBOOK 09 SUMMARY ARTIFACT (for Notebook 17's rollup)
# =============================================================================
_section("SECTION 13: Write Notebook 09 Summary Artifact")

notebook_09_summary = {
    "notebook": "09_mlops", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "champion_model": CHAMPION_NAME, "champion_model_version": _new_version, "champion_model_sha256": CHAMPION_MODEL_SHA256,
    "latency_summary": latency_summary,
    "output_files": {p.name: str(p) for p in _expected_files + [performance_report_path]},
}
nb09_summary_path = ARTIFACTS_DIR / "notebook_09_summary.json"
with open(nb09_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_09_summary, f, indent=2)
print(f"\u2705 Saved -> {nb09_summary_path}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 14: Notebook 09 Complete -- Handoff to Notebook 10")

print("NOTEBOOK 09: MLOPS -- COMPLETE")
print(f"  Champion registered              : {CHAMPION_NAME} v{_new_version}")
print(f"  Inference p99 latency (measured) : {latency_summary['single_row_p99_ms']:.2f} ms")
print(f"  Batch throughput (measured)      : {latency_summary['batch_throughput_rows_per_sec']:,.0f} rows/sec")
print(f"  Files produced                   : {len(_expected_files) + 2}")
for _p in _expected_files + [performance_report_path, nb09_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                    : 10_fastapi_deployment.ipynb")
print("\n\u2705 Ready to proceed.")
